### Middleware

In [12]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- long-running conversations that exceed context windows.
- multi-turn dialogues with extensive history
- applications where preserving full conversation context matters.

### Based on length of the message

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

## Message based summarization
agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("messages",5),
            keep=("messages",3)
        )
    ]
)

In [2]:
### Run with thread id
config={
    "configurable":{
        "thread_id":"test-1"
    }
}

In [ ]:
# Alternative test data
questions = [
    "what is 2+2",
    "what is 10*5",
    "what is 100/4",
    "What is 7*8?",
    "What is 20+30?",
    "What is 50/5?",
]

for q in questions:
    response = agent.invoke(
        {
            "messages":[HumanMessage(content=q)]
        },
        config
    )
    last_message = response["messages"][-1]
    print("Question:", q)
    print("Answer:", last_message.content[0]["text"])
    print(f"Messages: {len(response['messages'])}")

### Based on token size

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $100/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("tokens",550),
            keep=("tokens", 200)
        )
    ]
)

config = {
    "configurable":{
        "thread_id": "hotel-test-2"
    }
}

# Token Counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [14]:
# Run test
cities = ["Paris", "London"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Find hotels in {city}"
                )
            ]
        },
        config = config
    )
    tokens = count_tokens(response["messages"])
    last_message = response["messages"][-1]
    print(f"\n{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"Answer: {last_message.content}")
    


Paris: ~159 tokens, 4 messages
Answer: Here are some hotels available in Paris:

1. **Grand Hotel**
   - **Rating:** 5 stars
   - **Price:** $350/night
   - **Amenities:** Spa, pool, gym

2. **City Inn**
   - **Rating:** 4 stars
   - **Price:** $100/night
   - **Amenities:** Business center

3. **Budget Stay**
   - **Rating:** 3 stars
   - **Price:** $75/night
   - **Amenities:** Free wifi

Let me know if you would like more specific recommendations or help with anything else!

London: ~1567 tokens, 5 messages
Answer: Here are some hotel options in London:

1. **Grand Hotel** - 5-star, $350/night, amenities include spa, pool, and gym.
2. **City Inn** - 4-star, $100/night, amenities include a business center.
3. **Budget Stay** - 3-star, $75/night, amenities include free wifi.

Would you like more specific recommendations? If so, please share your preferred dates, budget range, neighborhood, or any other specific requirements.


### Human In the Loop MiddleWare

Pause agent execution for human approval, editing or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval(e.g. database writes, financial transactions)
- Complicance workflows where human oversight is mandatory.
- Long-running conversations where feedback guides the agent.

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [22]:
agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":[
                        "approve",
                        "edit",
                        "reject"
                    ]
                },
                "read_email_tool": False
            }
        )
    ]
)

In [23]:
config = {
    "configurable":{
        "thread_id": "test-approve"
    }
}
# step 1: request
result = agent.invoke(
    {
        "messages": [HumanMessage(
            content="Send email to john@test.com with subject 'hello' and body 'how are you'"
        )]
    },
    config = config
)

In [24]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'how are you'", additional_kwargs={}, response_metadata={}, id='7de8dcb2-3332-4d95-ad54-d196a2155925'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email.\nI have the necessary details: recipient (\'john@test.com\'), subject (\'hello\'), and body (\'how are you\').\nI should use the `send_email_tool`.\nI will call the tool with the provided arguments.\nThen I will return the result to the user.\nNo other information is needed.\nProceeding with the tool call. \nTool: send_email_tool\nArguments: recipient="john@test.com", subject="hello", body="how are you"\nDone. \nChecking tool definition:\nname: send_email_tool\nparameters: recipient (string), subject (string), body (string)\nAll match.\nCalling tool. \nWait, I\'ll just generate the tool call.\n', 'tool_calls': [{'id': 'fp5p4n4ym', 'function': {'arguments': '{"body":"how are you","recipient":"joh

In [25]:
# Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "approve"
                    }
                ]
            }
        ),
        config = config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been sent to john@test.com with the subject 'hello'.
